# 🚀 10주차 실습 (2026-06-12) - Theme 41: LangChain RAG(Retrieval-Augmented Generation) 시스템 기초

이번 실습에서는 외부 문서를 로드하고 분할하여 임베딩한 뒤, 벡터스토어에 저장하고 이를 LLM과 결합하여 질문에 대답하는 **RAG(검색 증강 생성) 시스템**의 핵심 파이프라인을 구축합니다.

실무 수준의 RAG를 이해하기 위해 가장 핵심이 되는 4가지 컴포넌트인 **Document Loader & Text Splitter**, **Embeddings**, **Vector Store**, **Retriever Chain**을 직접 코드로 작성해 보며 동작 흐름을 마스터합니다.

In [1]:
import os
from pathlib import Path
from typing import List, Literal
from getpass import getpass
from pydantic import BaseModel, Field

import pandas as pd
import numpy as np
from config import GOOGLE_AI_API_KEY

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.runnables.history import RunnableWithMessageHistory
# LangChain core vectorstores & text splitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

# Embedding model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

CHAT_MODEL = 'google_genai:gemma-4-31b-it'
# CHAT_MODEL = 'gemini-3.1-flash-lite'
print("실습 환경 준비 완료!")

실습 환경 준비 완료!


In [5]:
# Chat Model 초기화
model = init_chat_model(CHAT_MODEL, api_key=GOOGLE_AI_API_KEY)

# Embedding Model 초기화 (Google GenAI Embeddings 사용)
# models/embedding-001은 Gemini Developer API에서 가장 안정적으로 지원하는 텍스트 임베딩 모델입니다.
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2", 
    google_api_key=GOOGLE_AI_API_KEY
)
print("모델 및 임베딩 초기화 완료!")

모델 및 임베딩 초기화 완료!


## 1. Document Loading & Text Splitting (문서 로드 및 분할)

RAG 시스템의 첫 번째 단계는 분석할 외부 데이터를 가져와 LLM의 컨텍스트 윈도우 한계를 고려해 작은 크기(Chunk)로 쪼개는 것입니다.

이번 실습에서는 가상의 회사 규정 및 안내 텍스트를 대상 문서로 설정하고, `RecursiveCharacterTextSplitter`를 사용해 의미가 깨지지 않는 최적의 단위로 분할해 봅니다.

In [6]:
# 실습용 가상 문서 데이터 정의 (사내 복지 및 보안 규정 안내)
sample_document_text = """
[SK FAMILY AI 캠프 사내 복지 및 보안 규정 안내]

제1조 (목적)
본 규정은 SK FAMILY AI 캠프 임직원들의 건강 증진 및 쾌적한 연구 환경 제공, 그리고 정보 자산 보호를 목적으로 합니다.

제2조 (유연 근무제 및 휴식)
1. 캠프의 공식 코어 타임은 오전 10시부터 오후 4시까지입니다. 이 시간 동안은 부서 간 협업을 위해 반드시 근무 또는 소통 가능한 상태를 유지해야 합니다.
2. 주 40시간 근무 내에서 출퇴근 시간을 자유롭게 조정할 수 있습니다.
3. 1일 8시간 근무 시 최소 1시간의 휴게 시간이 보장되며, 연구동 3층 힐링존에 있는 안마의자와 수면실은 언제든 자유롭게 이용할 수 있습니다.

제3조 (건강 및 복지 지원)
1. 회사는 모든 임직원에게 연 1회 종합 건강검진을 전액 지원합니다.
2. 본인 및 배우자, 그리고 직계 비속의 의료비가 연 500만 원을 초과할 경우, 초과분의 80%를 지원받을 수 있습니다 (연 최대 한도 2,000만 원).
3. 자기개발비(도서 구매, 헬스장 등록, 학원 수강 등) 명목으로 매월 20만 원 상당의 복지 포인트를 지급합니다.

제4조 (정보 보안 및 출입 규정)
1. 모든 업무용 노트북은 외부 반출 시 사전 보안 승인이 필요합니다.
2. 연구동 및 사무동 출입 시 반드시 개인 사원증을 태그해야 하며, 외부인의 경우 1층 안내 데스크에서 방문 임시증을 발급받아 동행해야 합니다.
3. 비밀번호는 최소 8자리 이상으로 설정하고, 영문 대소문자, 숫자, 특수문자를 혼용해야 하며, 3개월 주기로 변경해야 합니다.
4. 기밀 업무 문서는 사내 파일 서버(Shared-NAS)에만 저장해야 하며, 개인 클라우드(Dropbox, Google Drive 등)로의 업로드는 엄격히 금지됩니다.
""".strip()

# Text Splitter 설정
# chunk_size: 쪼갤 최대 문자 수
# chunk_overlap: 문맥 끊김을 방지하기 위해 중첩시킬 문자 수
# separators: 줄바꿈, 온점 등을 기준으로 의미 단위를 유지하며 쪼갭니다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=["\n\n", "\n", " ", ""]
)

# 텍스트 분할 실행
chunks = text_splitter.split_text(sample_document_text)
print(f"원본 텍스트가 {len(chunks)}개의 조각으로 분할되었습니다.\n")

# 분할된 첫 3개 조각 출력 확인
for i, chunk in enumerate(chunks[:3]):
    print(f"--- [Chunk {i+1}] (길이: {len(chunk)}) ---")
    print(chunk)
    print()

원본 텍스트가 8개의 조각으로 분할되었습니다.

--- [Chunk 1] (길이: 118) ---
[SK FAMILY AI 캠프 사내 복지 및 보안 규정 안내]

제1조 (목적)
본 규정은 SK FAMILY AI 캠프 임직원들의 건강 증진 및 쾌적한 연구 환경 제공, 그리고 정보 자산 보호를 목적으로 합니다.

--- [Chunk 2] (길이: 107) ---
제2조 (유연 근무제 및 휴식)
1. 캠프의 공식 코어 타임은 오전 10시부터 오후 4시까지입니다. 이 시간 동안은 부서 간 협업을 위해 반드시 근무 또는 소통 가능한 상태를 유지해야 합니다.

--- [Chunk 3] (길이: 124) ---
2. 주 40시간 근무 내에서 출퇴근 시간을 자유롭게 조정할 수 있습니다.
3. 1일 8시간 근무 시 최소 1시간의 휴게 시간이 보장되며, 연구동 3층 힐링존에 있는 안마의자와 수면실은 언제든 자유롭게 이용할 수 있습니다.



## 2. Text Embeddings & Vector Store (텍스트 임베딩 및 벡터스토어)

분할된 텍스트 조각들을 벡터로 변환하여 의미적 유사도를 비교할 수 있도록 만듭니다. 

이번 실습에서는 별도의 데이터베이스 인프라 구축 없이 표준 라이브러리 규격만으로 빠르게 로컬 캐싱이 가능한 `InMemoryVectorStore`를 활용하여 벡터 데이터베이스 적재 과정을 실습합니다.

In [7]:
# InMemoryVectorStore 생성 및 분할된 텍스트 청크 추가
# add_texts 메서드를 통해 텍스트들이 자동으로 Google 임베딩 모델을 거쳐 벡터로 저장됩니다.
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_texts(chunks)

print("벡터스토어에 문서 조각 저장 완료!")

벡터스토어에 문서 조각 저장 완료!


## 3. Similarity Search (유사도 검색)

사용자의 질문(Query)이 들어오면 해당 질문 또한 동일한 임베딩 모델로 벡터화합니다. 이후 벡터 데이터베이스 내 저장된 조각들과 코사인 유사도를 계산하여 문맥상 질문과 가장 밀접한 문서 조각을 검색합니다.

In [8]:
query = "복지 포인트는 한 달에 얼마씩 나오나요?"

# 유사도 검색 실행 (가장 유사한 문서 2개 추출)
# similarity_search는 기본적으로 코사인 유사도 기반으로 가장 가깝고 의미가 통하는 문서를 찾아냅니다.
related_docs = vector_store.similarity_search(query, k=2)

print(f"질문: '{query}'\n")
for i, doc in enumerate(related_docs):
    print(f"📌 [유사 문서 {i+1}] (내용)")
    print(doc.page_content)
    print("-" * 40)

질문: '복지 포인트는 한 달에 얼마씩 나오나요?'

📌 [유사 문서 1] (내용)
3. 자기개발비(도서 구매, 헬스장 등록, 학원 수강 등) 명목으로 매월 20만 원 상당의 복지 포인트를 지급합니다.
----------------------------------------
📌 [유사 문서 2] (내용)
제3조 (건강 및 복지 지원)
1. 회사는 모든 임직원에게 연 1회 종합 건강검진을 전액 지원합니다.
2. 본인 및 배우자, 그리고 직계 비속의 의료비가 연 500만 원을 초과할 경우, 초과분의 80%를 지원받을 수 있습니다 (연 최대 한도 2,000만 원).
----------------------------------------


## 4. LCEL RAG Chain (검색 증강 생성 체인 구축)

검색기(Retriever)와 프롬프트 템플릿, 그리고 LLM을 하나의 유기적 파이프라인으로 연결하여 완성형 RAG 시스템을 구축합니다.

**동작 메커니즘**:
1. 사용자 질문을 받으면 `retriever`가 벡터스토어에서 관련 문서들을 찾습니다.
2. 찾아낸 문서 조각들을 프롬프트의 `{context}`에 채워넣고, 질문을 `{question}`에 결합합니다.
3. 결합된 프롬프트를 LLM에 넘겨, 컨텍스트에 기록된 사실을 바탕으로 정답을 반환하게 유도합니다 (할루시네이션 제어).

In [ ]:
# 1. Retriever 설정 (Vector Store를 검색기 인터페이스로 변환)
# search_kwargs={"k": 2}를 통해 항상 상위 2개의 유사 문서만 컨텍스트로 주입하도록 제어합니다.
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# 2. RAG용 프롬프트 템플릿 설계
# LLM이 주어진 컨텍스트에만 기반하여 사실에 입각한 답변을 하도록 지시합니다.
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 SK FAMILY AI 캠프의 친절한 사내 규정 안내 AI 비서입니다. 
반드시 아래 제공된 [Context]를 바탕으로 사용자의 질문에 답변해 주세요. 
만약 주어진 [Context] 내용만으로 답변을 알 수 없거나 확실하지 않다면, 억지로 지어내지 말고 \"제공된 규정 안내 문서만으로는 해당 내용을 확인할 수 없습니다.\"라고 정직하게 답변해 주세요.

[Context]
{context}"""),
    ("human", "질문: {question}")
])

# 3. 문서 객체 리스트를 하나의 문자열로 결합하는 포맷터 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. LCEL 체인 결합
# - 입력 질문 'question'을 그대로 전달하면서, 동시에 retriever를 통해 'context'에 관련 문서를 바인딩합니다.
# - format_docs 함수를 RunnableLambda나 체인 흐름에 녹여내어 문서 리스트를 문자열로 변환합니다.
rag_chain = (
    {
        "context": retriever | RunnablePassthrough(format_docs), 
        "question": RunnablePassthrough()
    }
    | rag_prompt 
    | model 
    | StrOutputParser()
)

print("RAG 체인 구축 성공!")

RAG 체인 구축 성공!


In [ ]:
# InMemoryVectorStore 생성 및 분할된 텍스트 청크 추가
# add_texts 메서드를 통해 텍스트들이 자동으로 임베딩 모델을 거쳐 벡터로 저장됩니다.
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_texts(chunks)

print("벡터스토어에 문서 조각 저장 완료!")

query = "복지 포인트는 한 달에 얼마씩 나오나요?"

# 유사도 검색 실행 (가장 유사한 문서 2개 추출)
# similarity_search는 기본적으로 코사인 유사도 기반으로 가장 가깝고 의미가 통하는 문서를 찾아냅니다.
related_docs = vector_store.similarity_search(query, k=2)

print(f"질문: '{query}'\n")
for i, doc in enumerate(related_docs):
    print(f"📌 [유사 문서 {i+1}] (내용)")
    print(doc.page_content)
    print("-" * 40)

In [ ]:
# 1. Retriever 설정 (Vector Store를 검색기 인터페이스로 변환)
# search_kwargs={"k": 2}를 통해 항상 상위 2개의 유사 문서만 컨텍스트로 주입하도록 제어합니다.
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# 2. RAG용 프롬프트 템플릿 설계
# LLM이 주어진 컨텍스트에만 기반하여 사실에 입각한 답변을 하도록 지시합니다.
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 SK FAMILY AI 캠프의 친절한 사내 규정 안내 AI 비서입니다. 
반드시 아래 제공된 [Context]를 바탕으로 사용자의 질문에 답변해 주세요. 
만약 주어진 [Context] 내용만으로 답변을 알 수 없거나 확실하지 않다면, 억지로 지어내지 말고 "제공된 규정 안내 문서만으로는 해당 내용을 확인할 수 없습니다."라고 정직하게 답변해 주세요.

[Context]
{context}"""),
    ("human", "질문: {question}")
])

# 3. 문서 객체 리스트를 하나의 문자열로 결합하는 포맷터 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. LCEL 체인 결합
# - 입력 질문 'question'을 그대로 전달하면서, 동시에 retriever를 통해 'context'에 관련 문서를 바인딩합니다.
# - format_docs 함수를 RunnableLambda나 체인 흐름에 녹여내어 문서 리스트를 문자열로 변환합니다.
rag_chain = (
    {
        "context": retriever | RunnablePassthrough(format_docs), 
        "question": RunnablePassthrough()
    }
    | rag_prompt 
    | model 
    | StrOutputParser()
)

print("RAG 체인 구축 성공!")

In [11]:
# 1. 문서에 존재하는 정보에 대한 질문 테스트
print("=== 질문 1: 사내 규정에 존재하는 정보 ===")
q1 = "업무용 노트북을 외부로 가져가려면 어떻게 해야 하나요?"
response1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A:\n{response1}\n")

# 2. 문서에 존재하지 않는 정보에 대한 질문 테스트 (할루시네이션 방지 검증)
print("=== 질문 2: 사내 규정에 존재하지 않는 정보 (할루시네이션 방지) ===")
q2 = "회사 구내식당의 점심 운영 시간은 몇 시부터인가요?"
response2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A:\n{response2}\n")

=== 질문 1: 사내 규정에 존재하는 정보 ===
Q: 업무용 노트북을 외부로 가져가려면 어떻게 해야 하나요?
A:
업무용 노트북을 외부로 반출하시려면 **사전 보안 승인**을 받으셔야 합니다.

=== 질문 2: 사내 규정에 존재하지 않는 정보 (할루시네이션 방지) ===
Q: 회사 구내식당의 점심 운영 시간은 몇 시부터인가요?
A:
제공된 규정 안내 문서만으로는 해당 내용을 확인할 수 없습니다.



## 5. From Scratch로 구현하는 RAG 파이프라인 (LangChain 없이 구현)

이번에는 LangChain 프레임워크의 도움 없이, 순수 파이썬 표준 라이브러리, **NumPy**, 그리고 **Google GenAI SDK**만을 활용하여 RAG 파이프라인을 직접 바닥부터 구현합니다.

이를 통해 텍스트 분할, 임베딩 벡터 연산, 코사인 유사도 계산, 프롬프트 조립 등 RAG 시스템의 내부 작동 원리를 수학적/논리적으로 완전히 해부하고 이해합니다.

In [12]:
# 1. 네이티브 Google GenAI SDK 클라이언트 초기화
from google import genai
from google.genai import types

# config에서 로드한 API 키를 사용하여 Client를 직접 생성합니다.
client = genai.Client(api_key=GOOGLE_AI_API_KEY)

NATIVE_CHAT_MODEL = 'gemma-4-31b-it'
NATIVE_EMBED_MODEL = 'gemini-embedding-2'
print("네이티브 클라이언트 준비 완료!")

네이티브 클라이언트 준비 완료!


In [13]:
# 2. From Scratch 텍스트 분할 (Text Splitting) 구현
# 마침표를 기준으로 문장을 구분하여, 의미 맥락이 끊기지 않는 단위로 결합하는 custom 분할 함수를 직접 작성합니다.
def custom_split_text(text: str, chunk_size: int = 150, chunk_overlap: int = 30) -> List[str]:
    # 줄바꿈을 공백으로 평탄화한 후 온점(.)을 기준으로 문장을 쪼갭니다.
    sentences = text.replace('\n', ' ').split('. ')
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        
        # 잘려나간 마침표 복원
        if not sentence.endswith('.'):
            sentence += '.'
            
        # 현재 청크에 문장을 붙였을 때 제한 길이를 초과하는지 검사
        if len(current_chunk) + len(sentence) <= chunk_size:
            if current_chunk:
                current_chunk += " " + sentence
            else:
                current_chunk = sentence
        else:
            if current_chunk:
                chunks.append(current_chunk)
            current_chunk = sentence
            
    if current_chunk:
        chunks.append(current_chunk)
    return chunks

# 분할 실행 및 첫 3개 청크 확인
native_chunks = custom_split_text(sample_document_text, chunk_size=150)
print(f"From Scratch 분할 완료: {len(native_chunks)}개 청크 생성\n")
for idx, chunk in enumerate(native_chunks[:3]):
    print(f"--- [Chunk {idx+1}] (길이: {len(chunk)}) ---")
    print(chunk)
    print()

From Scratch 분할 완료: 7개 청크 생성

--- [Chunk 1] (길이: 139) ---
[SK FAMILY AI 캠프 사내 복지 및 보안 규정 안내]  제1조 (목적) 본 규정은 SK FAMILY AI 캠프 임직원들의 건강 증진 및 쾌적한 연구 환경 제공, 그리고 정보 자산 보호를 목적으로 합니다. 제2조 (유연 근무제 및 휴식) 1.

--- [Chunk 2] (길이: 131) ---
캠프의 공식 코어 타임은 오전 10시부터 오후 4시까지입니다. 이 시간 동안은 부서 간 협업을 위해 반드시 근무 또는 소통 가능한 상태를 유지해야 합니다. 2. 주 40시간 근무 내에서 출퇴근 시간을 자유롭게 조정할 수 있습니다. 3.

--- [Chunk 3] (길이: 139) ---
1일 8시간 근무 시 최소 1시간의 휴게 시간이 보장되며, 연구동 3층 힐링존에 있는 안마의자와 수면실은 언제든 자유롭게 이용할 수 있습니다. 제3조 (건강 및 복지 지원) 1. 회사는 모든 임직원에게 연 1회 종합 건강검진을 전액 지원합니다. 2.



In [14]:
# 3. From Scratch 임베딩 생성 및 인메모리 벡터 DB 구축
# SDK의 client.models.embed_content API를 호출하여 실수형 리스트 벡터를 획득하고,
# 텍스트와 NumPy 어레이 벡터 쌍을 딕셔너리 리스트 구조로 보관합니다.
native_vector_store = []

print("각 텍스트 청크 임베딩 생성 및 적재 중...")
for chunk in native_chunks:
    # Google GenAI SDK 임베딩 호출
    response = client.models.embed_content(
        model=NATIVE_EMBED_MODEL,
        contents=chunk
    )
    # 실수 배열 획득
    vector = response.embeddings[0].values
    
    native_vector_store.append({
        "text": chunk,
        "vector": np.array(vector)  # NumPy 벡터 연산을 위해 Array화
    })
    
print(f"\n총 {len(native_vector_store)}개의 텍스트-벡터 쌍 적재 완료!")
print(f"임베딩 차원 크기: {len(native_vector_store[0]['vector'])}차원")

각 텍스트 청크 임베딩 생성 및 적재 중...

총 7개의 텍스트-벡터 쌍 적재 완료!
임베딩 차원 크기: 3072차원


In [15]:
# 4. NumPy 기반 코사인 유사도(Cosine Similarity) 계산 및 수동 검색 구현
# 공식: Cosine Similarity = (A · B) / (||A|| * ||B||)
def compute_cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)

def native_similarity_search(query: str, k: int = 2) -> List[dict]:
    # 1. 쿼리 텍스트 벡터화
    query_response = client.models.embed_content(
        model=NATIVE_EMBED_MODEL,
        contents=query
    )
    query_vector = np.array(query_response.embeddings[0].values)
    
    # 2. 적재된 벡터스토어 내 청크들과 루프를 돌며 유사도 점수 산출
    scored_results = []
    for item in native_vector_store:
        score = compute_cosine_similarity(query_vector, item["vector"])
        scored_results.append({
            "text": item["text"],
            "score": score
        })
        
    # 3. 유사도 점수 기준 내림차순 정렬 후 상위 k개 슬라이싱
    scored_results.sort(key=lambda x: x["score"], reverse=True)
    return scored_results[:k]

# 수동 유사도 검색 테스트
search_query = "복지 포인트는 한 달에 얼마씩 나오나요?"
matched_results = native_similarity_search(search_query, k=2)

print(f"질문: '{search_query}'\n")
for i, res in enumerate(matched_results):
    print(f"📌 [유사 문서 {i+1}] (유사도 점수: {res['score']:.4f})")
    print(res["text"])
    print("-" * 40)

질문: '복지 포인트는 한 달에 얼마씩 나오나요?'

📌 [유사 문서 1] (유사도 점수: 0.6201)
본인 및 배우자, 그리고 직계 비속의 의료비가 연 500만 원을 초과할 경우, 초과분의 80%를 지원받을 수 있습니다 (연 최대 한도 2,000만 원). 3. 자기개발비(도서 구매, 헬스장 등록, 학원 수강 등) 명목으로 매월 20만 원 상당의 복지 포인트를 지급합니다.
----------------------------------------
📌 [유사 문서 2] (유사도 점수: 0.5539)
1일 8시간 근무 시 최소 1시간의 휴게 시간이 보장되며, 연구동 3층 힐링존에 있는 안마의자와 수면실은 언제든 자유롭게 이용할 수 있습니다. 제3조 (건강 및 복지 지원) 1. 회사는 모든 임직원에게 연 1회 종합 건강검진을 전액 지원합니다. 2.
----------------------------------------


In [16]:
# 5. From Scratch RAG 파이프라인 조립 및 답변 생성
# 검색 결과 컨텍스트와 질문을 F-string 프롬프트로 병합하고, SDK의 generate_content API를 직접 호출합니다.
def native_rag_pipeline(question: str) -> str:
    # 1. 유사 문서 k=2개 검색
    retrieved_chunks = native_similarity_search(question, k=2)
    
    # 2. 검색 문서 평탄화 및 조립
    context_str = "\n\n".join(item["text"] for item in retrieved_chunks)
    
    # 3. 수동 시스템 지침(System Instruction) 구성
    system_instruction = f"""당신은 SK FAMILY AI 캠프의 친절한 사내 규정 안내 AI 비서입니다. 
반드시 아래 제공된 [Context]를 바탕으로 사용자의 질문에 답변해 주세요. 
만약 주어진 [Context] 내용만으로 답변을 알 수 없거나 확실하지 않다면, 억지로 지어내지 말고 "제공된 규정 안내 문서만으로는 해당 내용을 확인할 수 없습니다."라고 정직하게 답변해 주세요.

[Context]
{context_str}"""
    
    # 4. LLM API 직접 호출
    response = client.models.generate_content(
        model=NATIVE_CHAT_MODEL,
        contents=f"질문: {question}",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.0  # 사실 관계 중심의 일관된 답변을 유도하기 위해 0.0 설정
        )
    )
    return response.text

# 6. 최종 RAG 파이프라인 검증 및 결과 비교
print("=== [From Scratch] 질문 1: 규정에 존재하는 정보 ===")
test_q1 = "업무용 노트북을 외부로 가져가려면 어떻게 해야 하나요?"
ans1 = native_rag_pipeline(test_q1)
print(f"Q: {test_q1}")
print(f"A:\n{ans1}\n")

print("=== [From Scratch] 질문 2: 규정에 존재하지 않는 정보 (할루시네이션 방지) ===")
test_q2 = "회사 구내식당의 점심 운영 시간은 몇 시부터인가요?"
ans2 = native_rag_pipeline(test_q2)
print(f"Q: {test_q2}")
print(f"A:\n{ans2}\n")

=== [From Scratch] 질문 1: 규정에 존재하는 정보 ===
Q: 업무용 노트북을 외부로 가져가려면 어떻게 해야 하나요?
A:
업무용 노트북을 외부로 반출하시려면 **사전 보안 승인**을 받으셔야 합니다.

=== [From Scratch] 질문 2: 규정에 존재하지 않는 정보 (할루시네이션 방지) ===
Q: 회사 구내식당의 점심 운영 시간은 몇 시부터인가요?
A:
제공된 규정 안내 문서만으로는 해당 내용을 확인할 수 없습니다.

